In [3]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, PowerTransformer, QuantileTransformer, RobustScaler, StandardScaler

# Busca la raiz del repo aunque el notebook se ejecute desde notebooks/.
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from utils import Modelo, basic_comb_train


# Usa una muestra relativamente chica para iterar rapido.
data = pd.read_csv(repo_root / "data" / "training.csv").sample(4000, random_state=42)

# Se deja Weight y EventId porque basic_comb_train usa Weight para AMS
# y saca Weight/EventId antes de entrenar.
X_train = data.drop(columns=["Label"]).replace(-999.0, np.nan)
Y_train = data["Label"]

# Pipelines mas normales.
pipelines = [
    (
        "median_standard",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
    ),
    (
        "median_robust",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
        ]),
    ),
    (
        "median_minmax",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", MinMaxScaler()),
        ]),
    ),

    # Pipelines un poco mas jugados.
    (
        "knn_standard",
        Pipeline([
            ("imputer", KNNImputer(n_neighbors=5)),
            ("scaler", StandardScaler()),
        ]),
    ),
    (
        "power_standard",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("power", PowerTransformer()),
            ("scaler", StandardScaler()),
        ]),
    ),
    (
        "quantile_robust",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("quantile", QuantileTransformer(output_distribution="normal", n_quantiles=200, random_state=42)),
            ("scaler", RobustScaler()),
        ]),
    ),
    (
        "pca_standard",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=0.95, random_state=42)),
        ]),
    ),
]

models = [
    Modelo(
        "logreg",
        LogisticRegression(max_iter=1500, class_weight="balanced"),
    ),
    Modelo(
        "random_forest",
        RandomForestClassifier(
            n_estimators=120,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced",
        ),
    ),
    Modelo(
        "gradient_boosting",
        GradientBoostingClassifier(random_state=42),
    ),
]

# Algunas combinaciones se omiten porque no suelen tener mucho sentido
# o agregan costo sin demasiado valor para esta prueba.
dont_train = [
    ("pca_standard", "random_forest"),
    ("pca_standard", "gradient_boosting"),
]

resultados = basic_comb_train(
    X_train=X_train,
    Y_train=Y_train,
    models=models,
    pipelines=pipelines,
    k_fold=3,
    path=repo_root / "modelos_prueba",
    dont_train=dont_train,
)

resumen = pd.DataFrame([
    {
        "pipeline": r["pipeline_name"],
        "modelo": r["model_name"],
        "ams": round(r["mean_ams"], 6),
        "f1": round(r["mean_f1"], 6),
        "accuracy": round(r["mean_accuracy"], 6),
        "precision": round(r["mean_precision"], 6),
        "recall": round(r["mean_recall"], 6),
        "saved_path": str(r["saved_path"]),
    }
    for r in resultados
])

resumen


Se skipea par (pca_standard, random_forest)
Se skipea par (pca_standard, gradient_boosting)


,pipeline,modelo,ams,f1,accuracy,precision,recall,saved_path
0,power_standard,random_forest,0.190896,0.708391,0.820500,0.794866,0.638889,/home/guille/Desktop/estudio/TAA/proyectos/taa...
1,quantile_robust,random_forest,0.189759,0.709654,0.820750,0.793565,0.641810,/home/guille/Desktop/estudio/TAA/proyectos/taa...
2,knn_standard,random_forest,0.189365,0.700616,0.814751,0.781128,0.635168,/home/guille/Desktop/estudio/TAA/proyectos/taa...
3,median_standard,random_forest,0.189245,0.708311,0.820000,0.792382,0.640372,/home/guille/Desktop/estudio/TAA/proyectos/taa...
4,median_robust,random_forest,0.189245,0.708311,0.820000,0.792382,0.640372,/home/guille/Desktop/estudio/TAA/proyectos/taa...
5,median_minmax,random_forest,0.189245,0.708311,0.820000,0.792382,0.640372,/home/guille/Desktop/estudio/TAA/proyectos/taa...
6,median_standard,gradient_boosting,0.187517,0.729294,0.823750,0.766641,0.695425,/home/guille/Desktop/estudio/TAA/proyectos/taa...
7,median_robust,gradient_boosting,0.187517,0.729294,0.823750,0.766641,0.695425,/home/guille/Desktop/estudio/TAA/proyectos/taa...
8,median_minmax,gradient_boosting,0.187517,0.729294,0.823750,0.766641,0.695425,/home/guille/Desktop/estudio/TAA/proyectos/taa...
9,power_standard,gradient_boosting,0.187051,0.728786,0.823500,0.766438,0.694665,/home/guille/Desktop/estudio/TAA/proyectos/taa...
